# 🌀 SIFE-LDM V4.0 — Physics-Informed Multimodal Diffusion

**Structured Intelligence Field Latent Diffusion Model**

This notebook implements the architectural reconstruction of SIFE-LDM v4.0. It utilizes **Dual-Modality Diffusion**:
1. **Vision:** Continuous Polar Diffusion (Gaussian on Amplitude, Wrapped Gaussian on Phase).
2. **Text:** Masked Discrete Diffusion (Iterative phase prediction for semantic tokens).

**Dataset:** `fashion_mnist` (Open / Non-gated). We generate synthetic captions from labels for multimodal training.

## 1️⃣ Setup Environment

In [ ]:
!pip install -q flax optax datasets transformers
import jax
import jax.numpy as jnp
import numpy as np
print(f'JAX Version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

## 2️⃣ Clone SIFE-LDM v4.0 Repository

In [ ]:
!rm -rf SIFE-LDM-
!git clone https://github.com/vicekha/SIFE-LDM-.git
%cd SIFE-LDM-
import sys
sys.path.append('.')

from sife.model import SIFELDM, SIFELDMConfig, get_loss
from sife.field import SIFEConfig
from sife.diffusion import GaussianDiffusion, MaskedDiffusion
from sife.optim import andi
from sife.tokenizer import Vocabulary, SIFETokenizer
from train import create_train_state, train_step_vision_jit, train_step_text_jit

## 3️⃣ Prepare Fashion-MNIST Dataset (Multimodal)

In [ ]:
from datasets import load_dataset

print("Loading Fashion-MNIST dataset...")
ds = load_dataset('fashion_mnist', split='train')

IMAGE_SIZE = 32 # Small for fast iteration
LABEL_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def collate_fn(batch):
    imgs = []
    texts = []
    for item in batch:
        # Upscale 28x28 to 32x32 and normalize
        img = np.array(item['image'].resize((IMAGE_SIZE, IMAGE_SIZE)).convert('RGB'))
        imgs.append(img)
        # Generate synthetic caption
        label_name = LABEL_NAMES[item['label']]
        texts.append(f"a photo of a {label_name}")
    
    imgs = np.array(imgs).astype(np.float32) / 255.0
    return {'image': imgs, 'text': texts}

# Setup Tokenizer
vocab = Vocabulary()
vocab.build_from_texts([f"a photo of a {name}" for name in LABEL_NAMES])
tokenizer = SIFETokenizer(vocab, max_len=16)

print(f"Vocab size: {len(vocab)}")

## 4️⃣ Initialize v4.0 Model

In [ ]:
config = SIFELDMConfig(
    vocab_size=len(vocab),
    max_seq_len=16,
    embed_dim=128,
    num_layers=4,
    num_heads=4,
    mlp_dim=256,
    patch_size=8,
    image_size=IMAGE_SIZE,
    physics_config=SIFEConfig(gamma=0.2, alpha=0.1)
)

rng = jax.random.PRNGKey(0)
state = create_train_state(config, rng)
print("Model Initialized.")

## 5️⃣ Training Loop

In [ ]:
BATCH_SIZE = 32
STEPS = 500

for step in range(STEPS):
    idx = np.random.randint(0, len(ds), BATCH_SIZE)
    batch_raw = [ds[int(i)] for i in idx]
    batch = collate_fn(batch_raw)
    tokenized = tokenizer(batch['text'])
    
    rng, v_rng, t_rng = jax.random.split(rng, 3)
    
    # Interleaved step
    state, v_loss = train_step_vision_jit(state, {'image': batch['image']}, v_rng, config=config)
    state, t_loss = train_step_text_jit(state, {'tokens': tokenized['input_ids'], 'mask': tokenized['mask']}, t_rng, config=config)
    
    if step % 50 == 0:
        print(f"Step {step} | Vision Loss: {v_loss:.4f} | Text Loss: {t_loss:.4f}")

## 6️⃣ Generation

In [ ]:
from inference import generate_vision, generate_text
import matplotlib.pyplot as plt

# Generate Image
rng, gen_rng = jax.random.split(rng)
model = SIFELDM(config)
gen_imgs = generate_vision(model, state.params, config, gen_rng, batch_size=1)

plt.imshow(gen_imgs[0])
plt.title("Generated Feature Field")
plt.axis('off')
plt.show()

# Generate Text
prompt = "a photo of"
start_ids = tokenizer([prompt], padding=False)['input_ids'][0]
gen_tokens = generate_text(model, state.params, config, gen_rng, start_tokens=start_ids[None, :], max_new_tokens=5)
print("Generated Caption:", vocab.decode(gen_tokens[0].tolist()))